# Results

This chapter reports the performance of the evaluated forecasting strategies under the common chronological split and matched data design. The main comparison is followed by analyses of negative transfer, robustness, source filtering, and selector behaviour.


In [ ]:
import os
from pathlib import Path

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in candidate_roots if (path / "Data" / "processed").exists())
MPLCONFIG_PATH = PROJECT_ROOT / ".cache" / "matplotlib"
MPLCONFIG_PATH.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_PATH))

try:
    import matplotlib
    matplotlib.use("Agg")

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "This notebook needs a clean Python 3 kernel with numpy, pandas, and matplotlib. "
        "If your current Anaconda kernel shows a numpy or pyarrow error, switch kernels and run again."
    ) from exc

from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
METHOD_TABLES_DIR = PROCESSED_DIR / "methodology" / "tables"
EDA_TABLES_DIR = PROCESSED_DIR / "eda" / "tables"
RESULTS_FIG_DIR = PROCESSED_DIR / "results" / "figures"
RESULTS_TABLES_DIR = PROCESSED_DIR / "results" / "tables"
RESULTS_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

EDA_TABLE_GROUPS = {
    "overview": {
        "corpus_overview.csv",
        "feature_missingness.csv",
    },
    "correlations": {
        "pooled_lag_correlations.csv",
        "site_best_lag_summary.csv",
        "site_lag_correlations.csv",
    },
    "similarity": {
        "source_target_best_matches.csv",
        "source_target_best_matches_train.csv",
        "source_target_similarity.csv",
        "source_target_similarity_train.csv",
    },
}

METHOD_TABLE_GROUPS = {
    "protocol": {
        "protocol_summary.csv",
        "feature_inventory.csv",
        "similarity_summary.csv",
        "transfer_configuration.csv",
    },
    "core_results": {
        "summary_long.csv",
        "summary_wide.csv",
        "gain_vs_no_tl.csv",
        "site_metric_long.csv",
        "site_metric_wide.csv",
        "best_model_by_site.csv",
        "publication_core_comparison.csv",
    },
    "selector": {
        "selector_search.csv",
        "selector_by_site.csv",
        "selector_allocation.csv",
        "selector_feature_importance.csv",
        "oracle_selector_by_site.csv",
    },
    "transfer": {
        "site_transfer_effects.csv",
        "transfer_effect_summary.csv",
        "cohort_definitions.csv",
        "cohort_assignments.csv",
        "cohort_pooled_summary.csv",
        "cohort_site_summary.csv",
        "negative_transfer_site_summary.csv",
        "negative_transfer_summary.csv",
        "negative_transfer_similarity_bins.csv",
    },
    "robustness": {
        "selector_ablation_summary.csv",
        "paired_significance_tests.csv",
        "cluster_bootstrap_significance.csv",
        "label_budget_detail.csv",
        "label_budget_summary.csv",
        "seed_stability_detail.csv",
        "seed_stability_summary.csv",
        "rolling_temporal_folds.csv",
        "rolling_temporal_detail.csv",
        "rolling_temporal_summary.csv",
        "publication_robustness_comparison.csv",
    },
}

EDA_TABLE_DIRS = {group: EDA_TABLES_DIR / group for group in EDA_TABLE_GROUPS}
METHOD_TABLE_DIRS = {group: METHOD_TABLES_DIR / group for group in METHOD_TABLE_GROUPS}

EDA_TABLE_PATHS = {
    filename: EDA_TABLE_DIRS[group] / filename
    for group, filenames in EDA_TABLE_GROUPS.items()
    for filename in filenames
}
METHOD_TABLE_PATHS = {
    filename: METHOD_TABLE_DIRS[group] / filename
    for group, filenames in METHOD_TABLE_GROUPS.items()
    for filename in filenames
}

MODEL_COLORS = {
    "Context-Aware Selective Learning": "#0B6E4F",
    "Deep Context LSTM (Always TL)": "#C97C00",
    "Deep Context LSTM (No TL)": "#6B7280",
    "Random Forest": "#2C7FB8",
    "Baseline LSTM (No TL)": "#9AA5B1",
    "Baseline LSTM (TL)": "#B56576",
    "Site-Hard Expert Selector": "#8D6A9F",
}

SITE_EFFECT_COLORS = {
    "Selective rescue": "#0B6E4F",
    "Persistent harm": "#C0392B",
    "Both improve": "#2C7FB8",
    "Always TL only": "#9C6644",
}

FEATURE_GROUP_COLORS = {
    "Recent local state": "#0B6E4F",
    "Similarity profile": "#2C7FB8",
    "Seasonality": "#C97C00",
    "Site history": "#8D6A9F",
    "External drivers": "#9C6644",
    "Other": "#6B7280",
}


def methodology_table_path(name: str) -> Path:
    if name in METHOD_TABLE_PATHS:
        return METHOD_TABLE_PATHS[name]
    for directory in METHOD_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return METHOD_TABLES_DIR / name


def eda_table_path(name: str) -> Path:
    if name in EDA_TABLE_PATHS:
        return EDA_TABLE_PATHS[name]
    for directory in EDA_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return EDA_TABLES_DIR / name


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def apply_thesis_style() -> None:
    plt.rcParams.update(
        {
            "figure.dpi": 160,
            "savefig.dpi": 300,
            "font.family": "DejaVu Serif",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.facecolor": "#FBFBF8",
            "figure.facecolor": "white",
            "axes.grid": True,
            "grid.color": "#D7DBDD",
            "grid.linewidth": 0.6,
            "grid.alpha": 0.6,
            "axes.labelcolor": "#1F2933",
            "xtick.color": "#1F2933",
            "ytick.color": "#1F2933",
            "axes.edgecolor": "#A7B0B8",
        }
    )


def save_and_display(fig: plt.Figure, filename: str, width: int = 920) -> Path:
    out_path = RESULTS_FIG_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(out_path), width=width))
    print(f"Saved: {out_path}")
    return out_path


def _humanize_feature(name: str) -> str:
    pretty = {
        "context_last_wtda": "Recent local groundwater depth",
        "month_sin": "Seasonality sin(month)",
        "month_cos": "Seasonality cos(month)",
        "wtda_mean": "Target well mean depth",
        "wtda_std": "Target well depth variability",
        "pr_a_mean": "Mean precipitation anomaly",
        "sm_a_mean": "Mean soil-moisture anomaly",
        "tsmp_wtda_std": "TSMP depth variability",
        "pumping_log_std": "Pumping variability",
        "lon_norm": "Longitude",
        "lat_norm": "Latitude",
        "sim_min": "Minimum source similarity",
        "sim_q90": "90th percentile similarity",
        "sim_q95": "95th percentile similarity",
        "sim_top3_mean": "Mean top-3 similarity",
        "sim_top5_mean": "Mean top-5 similarity",
        "sim_top10_mean": "Mean top-10 similarity",
        "train_rows": "Available training history",
    }
    if name in pretty:
        return pretty[name]
    if name.startswith("shared_last_"):
        remainder = name.removeprefix("shared_last_").replace("_", " ")
        return f"Shared recent {remainder}"
    if name.startswith("context_last_"):
        remainder = name.removeprefix("context_last_").replace("_", " ")
        return f"Recent local {remainder}"
    return name.replace("_", " ")


def _feature_group(name: str) -> str:
    if name.startswith("sim_") or "similarity" in name:
        return "Similarity profile"
    if name.startswith("month_"):
        return "Seasonality"
    if name.startswith("shared_last_") or name.startswith("context_last_"):
        return "Recent local state"
    if name.endswith("_mean") or name.endswith("_std") or name in {"train_rows", "max_observations"}:
        return "Site history"
    if any(token in name for token in ("pr_", "sm_", "tsmp_", "pumping", "lon_norm", "lat_norm")):
        return "External drivers"
    return "Other"


def _binned_mean(df: pd.DataFrame, xcol: str, ycol: str, n_bins: int = 6) -> pd.DataFrame:
    clean = df[[xcol, ycol]].dropna().copy()
    if clean.empty:
        return clean
    clean["bin"] = pd.qcut(clean[xcol], q=min(n_bins, clean[xcol].nunique()), duplicates="drop")
    grouped = (
        clean.groupby("bin", observed=True)
        .agg(x_mean=(xcol, "mean"), y_mean=(ycol, "mean"), n=(ycol, "size"))
        .reset_index(drop=True)
    )
    return grouped

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_FIG_DIR:", RESULTS_FIG_DIR)


## 1. Main Comparison

The first comparison focuses on held-out test performance under the fixed evaluation protocol.


In [ ]:
core_table = read_csv(methodology_table_path("publication_core_comparison.csv")).copy()
show_cols = [col for col in ["model_family", "rmse_test", "mae_test", "r2_test"] if col in core_table.columns]
core_display = core_table[show_cols].sort_values("rmse_test").reset_index(drop=True)
num_cols = core_display.select_dtypes(include=[np.number]).columns
core_display[num_cols] = core_display[num_cols].round(3)
display(core_display)


In [ ]:
apply_thesis_style()
core = read_csv(methodology_table_path("publication_core_comparison.csv")).copy()
core = core.sort_values("rmse_test", ascending=False)
colors = [MODEL_COLORS.get(model, "#6B7280") for model in core["model_family"]]
best_rmse = float(core["rmse_test"].min())

fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.barh(core["model_family"], core["rmse_test"], color=colors, edgecolor="white", linewidth=1.0)
ax.axvline(best_rmse, linestyle="--", linewidth=1.0, color="#1F2933", alpha=0.7)
ax.set_xlabel("Test RMSE")
ax.set_ylabel("")
fig.tight_layout()
main_comparison_path = save_and_display(fig, "results_main_comparison.png", width=900)


Under the common evaluation design, Context-Aware Selective Learning achieves the lowest test RMSE (`1.263`), ahead of fixed transfer (`1.295`), Random Forest (`1.298`), and the deep no-transfer model (`1.304`). The numerical gap over fixed transfer is modest, but I do not interpret this as only a ranking result. The more important finding is that transfer becomes most useful when I apply it selectively rather than as a fixed rule across all target wells.


## 2. Negative Transfer And Recovery

This section examines which wells benefit from transfer, which wells are harmed by transfer, and to what extent selective learning mitigates those harmful cases.


In [ ]:
apply_thesis_style()
transfer_summary = read_csv(methodology_table_path("transfer_effect_summary.csv")).copy()
plot_df = transfer_summary[transfer_summary["strategy"].isin(["always_tl", "selective_learning"])].copy()
plot_df["label"] = plot_df["strategy"].map(
    {
        "always_tl": "Always transfer",
        "selective_learning": "Selective learning",
    }
)

helped = plot_df["helped_wells"].to_numpy(dtype=float)
harmed = plot_df["harmed_wells"].to_numpy(dtype=float)
y_pos = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.barh(y_pos, helped, color="#0B6E4F", label="Improved vs No TL")
ax.barh(y_pos, harmed, left=helped, color="#C0392B", label="Harmed vs No TL")
ax.set_yticks(y_pos, plot_df["label"])
ax.set_xlabel("Number of target wells")
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
fig.subplots_adjust(bottom=0.22)
transfer_counts_path = save_and_display(fig, "results_transfer_help_harm_counts.png", width=900)


In the site-level help-harm comparison, fixed transfer improves `56` wells and harms `41`, whereas selective learning improves `68` wells and harms `29`. I therefore do not interpret transfer as a uniformly positive intervention in the Amsterdam setting. The gain from selective learning comes from changing the well-level decision, not from assuming that every target well should use transferred information.


In [ ]:
apply_thesis_style()
negative_transfer_summary = read_csv(methodology_table_path("negative_transfer_summary.csv")).copy()
harmed_subset = negative_transfer_summary.query(
    "analysis_unit == 'site' and subset_label == 'always_tl_harmed_sites'"
).iloc[0]
partial_rate = float(harmed_subset["selective_partial_rescue_rate"])
full_rate = float(harmed_subset["selective_full_rescue_rate"])

fig, ax = plt.subplots(figsize=(6.6, 4.8))
rescue_rates = np.array([partial_rate, full_rate]) * 100.0
rescue_labels = ["Partial rescue", "Full rescue"]
rescue_colors = ["#2C7FB8", "#0B6E4F"]
ax.bar(rescue_labels, rescue_rates, color=rescue_colors, width=0.58)
ax.set_ylim(0, 100)
ax.set_ylabel("Share of always-TL harmed wells (%)")
fig.tight_layout()
rescue_path = save_and_display(fig, "results_negative_transfer_rescue_rates.png", width=760)


Among the wells harmed by fixed transfer, selective learning partially rescues `90.7%` and fully rescues `41.9%`. I see this as one of the strongest results in the thesis, because it shows that selective transfer is useful mainly as a safeguard against negative transfer rather than as a blanket transfer rule.


In [ ]:
apply_thesis_style()
site_effects = read_csv(methodology_table_path("site_transfer_effects.csv")).copy()

def classify(row: pd.Series) -> str:
    x = float(row["delta_rmse_always_tl_vs_no_tl"])
    y = float(row["delta_rmse_selective_tl_vs_no_tl"])
    if x > 0 and y < 0:
        return "Selective rescue"
    if x > 0 and y >= 0:
        return "Persistent harm"
    if x <= 0 and y < 0:
        return "Both improve"
    return "Always TL only"

site_effects["regime"] = site_effects.apply(classify, axis=1)
site_effects["marker_size"] = np.clip(
    site_effects["months_used"].fillna(site_effects["months_used"].median()), 10, 110
) * 2.2

fig, ax = plt.subplots(figsize=(8.0, 6.3))
for regime, sub in site_effects.groupby("regime", observed=True):
    ax.scatter(
        sub["delta_rmse_always_tl_vs_no_tl"],
        sub["delta_rmse_selective_tl_vs_no_tl"],
        s=sub["marker_size"],
        alpha=0.8,
        color=SITE_EFFECT_COLORS[regime],
        edgecolor="white",
        linewidth=0.7,
        label=regime,
    )

bounds = np.array(
    [
        site_effects["delta_rmse_always_tl_vs_no_tl"].min(),
        site_effects["delta_rmse_always_tl_vs_no_tl"].max(),
        site_effects["delta_rmse_selective_tl_vs_no_tl"].min(),
        site_effects["delta_rmse_selective_tl_vs_no_tl"].max(),
    ]
)
pad = 0.03
lower, upper = bounds.min() - pad, bounds.max() + pad
ax.plot([lower, upper], [lower, upper], linestyle="--", color="#7F8C8D", linewidth=1.0, alpha=0.7)
ax.axhline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.axvline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.set_xlim(lower, upper)
ax.set_ylim(lower, upper)
ax.set_xlabel("Always TL - No TL (delta RMSE)")
ax.set_ylabel("Selective learning - No TL (delta RMSE)")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.subplots_adjust(bottom=0.24)
site_heterogeneity_path = save_and_display(fig, "results_site_transfer_heterogeneity.png", width=860)


The site-level comparison confirms strong heterogeneity. `51` wells improve under both transfer strategies, `18` are clear selective rescues, `25` remain harmed under both strategies, and `6` benefit only from fixed transfer. This heterogeneity explains why average test performance is not enough on its own for interpreting transfer effects.


## 3. Robustness

Robustness analyses are reported as thesis results rather than as supplementary checks. They are used to assess whether the advantage of selective learning persists under alternative resampling and evaluation settings.


In [ ]:
robustness_table = read_csv(methodology_table_path("publication_robustness_comparison.csv")).copy()
robustness_display = robustness_table.copy()
num_cols = robustness_display.select_dtypes(include=[np.number]).columns
robustness_display[num_cols] = robustness_display[num_cols].round(3)
display(robustness_display)

paired = read_csv(methodology_table_path("paired_significance_tests.csv")).copy()
paired = paired.query("model_a == 'Context-Aware Selective Learning'").copy()
paired_display = paired[[
    "model_b",
    "mean_delta_metric_a_minus_b",
    "wins_model_a",
    "wins_model_b",
    "p_value_holm",
]].copy()
paired_display.columns = [
    "comparison_model",
    "mean_abs_error_delta",
    "wins_selective",
    "wins_comparison",
    "holm_p_value",
]
paired_display["mean_abs_error_delta"] = paired_display["mean_abs_error_delta"].round(4)
paired_display["holm_p_value"] = paired_display["holm_p_value"].round(6)
display(paired_display)


In [ ]:
apply_thesis_style()
bootstrap = read_csv(methodology_table_path("cluster_bootstrap_significance.csv")).copy()
label_map = {
    "Context-Aware Selective Learning": "Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
bootstrap_df = bootstrap.query("model_a == 'Context-Aware Selective Learning'").copy()
bootstrap_df["comparison"] = bootstrap_df["model_b"].map(label_map).fillna(bootstrap_df["model_b"])
bootstrap_df = bootstrap_df.sort_values("mean_delta_site_rmse_a_minus_b")

y = np.arange(len(bootstrap_df))
mean_delta = bootstrap_df["mean_delta_site_rmse_a_minus_b"].to_numpy(dtype=float)
low = bootstrap_df["bootstrap_ci_lower"].to_numpy(dtype=float)
high = bootstrap_df["bootstrap_ci_upper"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.hlines(y, low, high, color="#2C7FB8", linewidth=2.2)
ax.scatter(mean_delta, y, color="#0B6E4F", s=48, zorder=3)
ax.axvline(0, color="#1F2933", linewidth=0.9, alpha=0.7)
ax.set_yticks(y, bootstrap_df["comparison"])
ax.set_xlabel("Mean site RMSE delta")
fig.tight_layout()
bootstrap_path = save_and_display(fig, "results_cluster_bootstrap.png", width=900)


The cluster bootstrap supports the site-level advantage of selective learning. For the comparison against fixed transfer, the mean site-RMSE delta remains negative and the bootstrap interval stays below zero. Together with the paired site-level tests, this suggests that the main advantage is not driven by a few influential wells.


In [ ]:
apply_thesis_style()
seed_summary = read_csv(methodology_table_path("seed_stability_summary.csv")).copy()
focus_models = [
    "Context-Aware Selective Learning",
    "Deep Context LSTM (Always TL)",
    "Deep Context LSTM (No TL)",
    "Random Forest",
]
label_map = {
    "Context-Aware Selective Learning": "Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
seed_df = seed_summary.query("split == 'test' and model_family in @focus_models").copy()
seed_df["label"] = seed_df["model_family"].map(label_map)
seed_df["label"] = pd.Categorical(seed_df["label"], categories=["Selective", "Always TL", "No TL", "Random Forest"], ordered=True)
seed_df = seed_df.sort_values("label")

y_pos = np.arange(len(seed_df))
fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.errorbar(
    seed_df["mean_rmse"],
    y_pos,
    xerr=seed_df["std_rmse"],
    fmt="o",
    color="#1F2933",
    ecolor="#7F8C8D",
    elinewidth=1.4,
    capsize=3,
)
ax.set_yticks(y_pos, seed_df["label"])
ax.set_xlabel("Test RMSE")
fig.tight_layout()
seed_path = save_and_display(fig, "results_seed_stability.png", width=820)


Across repeated random seeds, selective learning remains the strongest of the main models on the test split, with mean RMSE `1.276` and standard deviation `0.009`. This reduces the concern that the main result depends on one favourable initialisation.


In [ ]:
apply_thesis_style()
rolling_summary = read_csv(methodology_table_path("rolling_temporal_summary.csv")).copy()
focus_models = [
    "Context-Aware Selective Learning",
    "Deep Context LSTM (Always TL)",
    "Deep Context LSTM (No TL)",
    "Random Forest",
]
label_map = {
    "Context-Aware Selective Learning": "Selective",
    "Deep Context LSTM (Always TL)": "Always TL",
    "Deep Context LSTM (No TL)": "No TL",
    "Random Forest": "Random Forest",
}
rolling_df = rolling_summary.query("split == 'test' and model_family in @focus_models").copy()
rolling_df["label"] = rolling_df["model_family"].map(label_map)
rolling_df["label"] = pd.Categorical(rolling_df["label"], categories=["Selective", "Always TL", "No TL", "Random Forest"], ordered=True)
rolling_df = rolling_df.sort_values("label")

y_pos = np.arange(len(rolling_df))
fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.errorbar(
    rolling_df["mean_rmse"],
    y_pos,
    xerr=rolling_df["std_rmse"],
    fmt="o",
    color="#1F2933",
    ecolor="#7F8C8D",
    elinewidth=1.4,
    capsize=3,
)
ax.set_yticks(y_pos, rolling_df["label"])
ax.set_xlabel("Test RMSE")
fig.tight_layout()
rolling_path = save_and_display(fig, "results_rolling_temporal_stability.png", width=820)


Across rolling temporal folds, selective learning again has the lowest mean test RMSE (`1.251`). The spread across folds is still noticeable, which is expected in a chronological design, but the ranking remains consistent enough to support the main conclusion.


In [ ]:
apply_thesis_style()
ablation = read_csv(methodology_table_path("selector_ablation_summary.csv")).copy()
ablation = ablation.sort_values("rmse_test", ascending=False)

fig, ax = plt.subplots(figsize=(8.8, 4.9))
ax.barh(ablation["ablation_label"], ablation["rmse_test"], color="#2C7FB8", edgecolor="white", linewidth=1.0)
ax.set_xlabel("Test RMSE")
ax.set_ylabel("")
fig.tight_layout()
ablation_path = save_and_display(fig, "results_selector_ablation.png", width=980)


The ablation results show that the selector loses accuracy when key information blocks are removed. The best gate uses context, similarity, and target-state information together, whereas removing target-state or similarity information leads to a measurable deterioration in test RMSE.


## 4. Source Filtering

This section evaluates the sensitivity of the source-filtering rule used for transfer selection.


In [ ]:
apply_thesis_style()
best_matches = read_csv(eda_table_path("source_target_best_matches_train.csv")).copy()
full_similarity = read_csv(eda_table_path("source_target_similarity_train.csv")).copy()

thresholds = np.round(np.arange(0.20, 0.71, 0.05), 2)
n_wells = int(best_matches["site_id"].nunique())

records = []
for tau in thresholds:
    matched_wells = int((best_matches["cosine_similarity"] >= tau).sum())
    retained_cells = int(full_similarity.loc[full_similarity["cosine_similarity"] >= tau, "cell_id"].nunique())
    records.append(
        {
            "threshold": tau,
            "matched_wells": matched_wells,
            "matched_well_share": matched_wells / n_wells,
            "retained_source_cells": retained_cells,
        }
    )

grid = pd.DataFrame(records)

grid_display = grid.loc[grid['threshold'].isin([0.50, 0.55, 0.60, 0.65]), [
    'threshold', 'retained_source_cells', 'matched_wells', 'matched_well_share'
]].copy()
grid_display['matched_well_share'] = (grid_display['matched_well_share'] * 100).round(1)
grid_display.columns = ['threshold', 'retained_source_cells', 'matched_wells', 'matched_well_share_pct']
display(grid_display)

fig, ax1 = plt.subplots(figsize=(9.0, 4.8))
ax2 = ax1.twinx()
ax1.plot(grid["threshold"], grid["retained_source_cells"], color="#2C7FB8", marker="o", linewidth=2.2, label="Retained source cells")
ax2.plot(grid["threshold"], grid["matched_well_share"] * 100, color="#C97C00", marker="s", linewidth=2.0, label="Target wells with match (%)")
ax1.axvline(0.60, linestyle="--", linewidth=1.0, color="#1F2933", alpha=0.75)
ax1.set_xlabel("Similarity threshold for source filtering")
ax1.set_ylabel("Retained source cells", color="#2C7FB8")
ax2.set_ylabel("Target wells with qualifying match (%)", color="#C97C00")
lines = [
    Line2D([0], [0], color="#2C7FB8", marker="o", linewidth=2.2, label="Retained source cells"),
    Line2D([0], [0], color="#C97C00", marker="s", linewidth=2.0, label="Target wells with match (%)"),
]
ax1.legend(handles=lines, frameon=False, fontsize=8.5, loc="upper right")
fig.tight_layout()
source_filter_path = save_and_display(fig, "results_source_threshold_sensitivity.png", width=960)


The threshold grid shows a steep trade-off. At `0.50`, the filter retains `187` source cells and `15/99` target wells with a direct qualifying match; at `0.60`, it retains `88` source cells and `10/99` wells; at `0.65`, it falls to `51` source cells and `5/99` wells. I therefore use `maxsim0.60` as a conservative pretraining filter: it removes a large share of weak source cells without pretending that most Amsterdam wells have strong direct source analogues. In this thesis, the conclusion does not depend on treating `0.60` as a universal rule; instead, it is a defensible restriction before the selector makes the final transfer decision.


## 5. Selector Interpretation

Because the selector is the main methodological contribution, this section examines both which features are most influential and how the transfer decision changes with source-target similarity.


In [ ]:
apply_thesis_style()
selector_importance = read_csv(methodology_table_path("selector_feature_importance.csv")).copy()
top_features = selector_importance.nlargest(12, "importance").copy().sort_values("importance", ascending=True)
top_features["feature_label"] = top_features["feature"].map(_humanize_feature)
top_features["feature_group"] = top_features["feature"].map(_feature_group)
top_features["color"] = top_features["feature_group"].map(FEATURE_GROUP_COLORS)

fig, ax = plt.subplots(figsize=(8.8, 5.3))
ax.barh(
    top_features["feature_label"],
    top_features["importance"],
    color=top_features["color"],
    edgecolor="white",
    linewidth=1.0,
)
ax.set_xlabel("Selector feature importance")
feature_handles = [
    Patch(facecolor=color, edgecolor="none", label=group)
    for group, color in FEATURE_GROUP_COLORS.items()
    if group in set(top_features["feature_group"])
]
ax.legend(handles=feature_handles, frameon=False, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.subplots_adjust(bottom=0.24)
selector_feature_path = save_and_display(fig, "results_selector_feature_importance.png", width=980)


Feature-importance results show that the selector relies most strongly on recent local groundwater depth, seasonality, site-history summaries, and a small group of shared-source signals. I interpret this as evidence that the gate does not choose transfer from similarity alone. It combines local target-state information with broader source-target context.


In [ ]:
apply_thesis_style()
selector_by_site = read_csv(methodology_table_path("selector_by_site.csv")).copy()
selector_by_site["decision_group"] = selector_by_site["selected_parent"].fillna("Deep Context LSTM (No TL)").map(
    {
        "Deep Context LSTM (Always TL)": "Transfer-heavy decision",
        "Deep Context LSTM (No TL)": "No-transfer decision",
        "Random Forest": "Fallback baseline",
    }
).fillna("Fallback baseline")

decision_colors = {
    "Transfer-heavy decision": "#C97C00",
    "No-transfer decision": "#6B7280",
    "Fallback baseline": "#2C7FB8",
}

fig, ax = plt.subplots(figsize=(8.2, 5.2))
for decision_group, sub in selector_by_site.groupby("decision_group", observed=True):
    ax.scatter(
        sub["cosine_similarity"],
        sub["selector_gamma_mean"],
        s=np.clip(sub["months_used"].fillna(sub["months_used"].median()), 5, 110) * 2.5,
        alpha=0.75,
        color=decision_colors[decision_group],
        edgecolor="white",
        linewidth=0.6,
        label=decision_group,
    )

binned = _binned_mean(selector_by_site, "cosine_similarity", "selector_gamma_mean")
if not binned.empty:
    ax.plot(binned["x_mean"], binned["y_mean"], color="#1F2933", linewidth=2.0, linestyle="--")

ax.set_xlabel("Best source-target cosine similarity")
ax.set_ylabel("Mean selector gamma")
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
fig.tight_layout()
selector_decision_path = save_and_display(fig, "results_selector_decision_pattern.png", width=900)


The decision plot reinforces that interpretation. Higher source-target similarity is compatible with more transfer-heavy behaviour, but the overlap across decision groups is substantial. I therefore do not read the selector as a hidden threshold rule. It uses similarity as one cue, while recent local state and site history still shape the final choice.


## 6. Discussion And Limitations

Taken together, these results support selective transfer as the strongest option within the present Amsterdam design. The advantage over fixed transfer is real but moderate in aggregate. I therefore frame the contribution of this thesis as a decision result: transfer is useful, but it should not be imposed as a fixed rule across all target wells.

The negative-transfer analysis is central for this interpretation. Fixed transfer helps many wells, but it also harms a substantial minority. Selective learning improves the overall ranking mainly because it reduces these harmful cases and rescues many of the wells that fixed transfer would otherwise degrade.

I still interpret these findings within three main limits. First, the Amsterdam target set is restricted, so I do not claim automatic generalisation to other Dutch regions or other hydrogeological settings. Second, the source and target domains are only partially aligned, which is exactly why a conservative source filter and a selector are necessary in this study. Third, monthly aggregation smooths short-term dynamics, so the reported gains apply to monthly forecasting and may not transfer directly to higher-frequency prediction tasks.

The robustness checks reduce the risk that the main ranking is driven by one seed or one evaluation window, but they do not replace external validation on a new region. For that reason, I present this thesis as evidence that selective transfer is the better framework for this Amsterdam case study, not as proof that the same ranking will hold unchanged in every groundwater forecasting problem.
